In [ ]:
# 窗口1：导入库、随机种子、全局参数
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import HeteroData
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from datetime import datetime
from tqdm import tqdm
import matplotlib.pyplot as plt
import random
import os

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# ---------- 2. 全局参数 ----------
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEQ_LEN = 3
PRED_HORIZON = 1
BATCH_SIZE = 8
EPOCHS = 50
HIDDEN_CHANNELS = 8          # 超图卷积后的隐层维度
NUM_HYPEREDGE_FEAT = 9       # 超边特征维度（F1-F9）
PATIENCE = 10
LEARNING_RATE = 0.001

LOSS_LOG_PATH = './Hyper2/loss_log_hyper6.csv'
METRICS_LOG_PATH = './Hyper2/metrics_log_hyper6.csv'
LOSS_CURVE_PATH = './Hyper2/loss_curve_hyper6.png'
METRICS_CURVE_PATH = './Hyper2/metrics_curve_hyper6.png'
MODEL_SAVE_PATH = './Hyper2/best_model_hyper6.pth'

# ---------- 3. 加载数据 ----------
scenic_static = pd.read_csv('./data/scenic_static.csv')
hotel_static = pd.read_csv('./data/hotel_static.csv')
nodes = pd.read_csv('./data/nodes.csv')
hotel_friction_daily = pd.read_csv('./data/hotel_friction_daily.csv')
poi_friction = pd.read_csv('./data/poi_friction.csv')          # 静态POI摩擦 (F1,F3,F4)
scenic_friction = pd.read_csv('./data/scenic_friction1-9.csv')
weather_daily = pd.read_csv('./data/weather_daily.csv')
search_daily = pd.read_csv('./data/search_daily.csv')
scenic_visit_daily = pd.read_csv('./data/scenic_visit_daily.csv')

# 统一日期格式
for df in [hotel_friction_daily, scenic_friction, weather_daily, search_daily, scenic_visit_daily]:
    df['date'] = pd.to_datetime(df['date'])

# ---------- 4. 确定有效景区 ----------
valid_visit_scenic = set(scenic_visit_daily['scenic_id'].unique())
exclude_scenic = {'s08', 's24', 's37'}
valid_scenic_ids = list(valid_visit_scenic - exclude_scenic)
print(f"有效景区数量: {len(valid_scenic_ids)}")
scenic_static = scenic_static[scenic_static['scenic_id'].isin(valid_scenic_ids)].copy()

# ---------- 5. 节点空间筛选（地理邻近）----------
def haversine_np(lon1, lat1, lon2, lat2):
    lon1 = np.asarray(lon1).reshape(-1, 1)
    lat1 = np.asarray(lat1).reshape(-1, 1)
    lon2 = np.asarray(lon2).reshape(1, -1)
    lat2 = np.asarray(lat2).reshape(1, -1)
    R = 6371
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

scenic_lon = scenic_static['lon'].values[:, None]
scenic_lat = scenic_static['lat'].values[:, None]

# 酒店筛选 (≤2km)
hotel_lon = hotel_static['lon'].values[:, None]
hotel_lat = hotel_static['lat'].values[:, None]
dist_hotel = haversine_np(hotel_lon, hotel_lat, scenic_lon, scenic_lat)
hotel_nodes = hotel_static[(dist_hotel <= 2.0).any(axis=1)].copy()

# POI筛选 (≤1.5km)
pois = nodes[nodes['node_type'] == 'poi'].copy()
poi_lon = pois['lon'].values[:, None]
poi_lat = pois['lat'].values[:, None]
dist_poi = haversine_np(poi_lon, poi_lat, scenic_lon, scenic_lat)
poi_nodes = pois[(dist_poi <= 1.5).any(axis=1)].copy()

print(f"酒店节点数: {len(hotel_nodes)}，POI节点数: {len(poi_nodes)}")

# 分配本地索引
scenic_static['node_idx'] = range(len(scenic_static))
hotel_nodes['node_idx'] = range(len(hotel_nodes))
poi_nodes['node_idx'] = range(len(poi_nodes))

# 节点ID到 (type, local_idx) 的映射
node_id_to_idx = {}
for _, row in scenic_static.iterrows():
    node_id_to_idx[row['scenic_id']] = ('scenic', row['node_idx'])
for _, row in hotel_nodes.iterrows():
    node_id_to_idx[row['hotel_id']] = ('hotel', row['node_idx'])
for _, row in poi_nodes.iterrows():
    node_id_to_idx[row['node_id']] = ('poi', row['node_idx'])

# ---------- 6. 静态特征编码（节点特征）----------
grade_encoder = LabelEncoder()
scenic_static['grade_code'] = grade_encoder.fit_transform(scenic_static['grade'])
scenic_static_feat = torch.tensor(scenic_static[['grade_code']].values, dtype=torch.float)

hotel_nodes['star_code'] = hotel_nodes['star'].fillna('3').astype(str)
star_encoder = LabelEncoder()
hotel_nodes['star_code'] = star_encoder.fit_transform(hotel_nodes['star_code'])
hotel_static_feat = torch.tensor(hotel_nodes[['star_code']].values, dtype=torch.float)

poi_cat_encoder = LabelEncoder()
poi_nodes['cat_code'] = poi_cat_encoder.fit_transform(poi_nodes['poi_category'])
poi_static_feat = torch.tensor(poi_nodes[['cat_code']].values, dtype=torch.float)

# 预计算每个景区的周边酒店索引和周边POI索引（地理距离筛选）
scenic_id_to_hotel_indices = {}
scenic_id_to_poi_indices = {}
for _, srow in scenic_static.iterrows():
    scenic_id = srow['scenic_id']
    scenic_lon, scenic_lat = srow['lon'], srow['lat']
    # 周边酒店（距离≤2km）
    hotels_near = []
    for _, hrow in hotel_nodes.iterrows():
        dist = haversine_np(np.array([scenic_lon]), np.array([scenic_lat]),
                            np.array([hrow['lon']]), np.array([hrow['lat']]))[0,0]
        if dist <= 2.0:
            hotels_near.append(hrow['node_idx'])
    scenic_id_to_hotel_indices[scenic_id] = hotels_near
    # 周边POI（距离≤1.5km）
    pois_near = []
    for _, prow in poi_nodes.iterrows():
        dist = haversine_np(np.array([scenic_lon]), np.array([scenic_lat]),
                            np.array([prow['lon']]), np.array([prow['lat']]))[0,0]
        if dist <= 1.5:
            pois_near.append(prow['node_idx'])
    scenic_id_to_poi_indices[scenic_id] = pois_near

print("静态地理邻近关系构建完成")

# ---------- 7. 时间划分 ----------
all_dates_in_data = scenic_friction['date'].unique()
all_dates_in_data = pd.Series(all_dates_in_data).sort_values()
start_date = all_dates_in_data.min()
end_date = all_dates_in_data.max()
print(f"数据日期范围: {start_date.date()} 至 {end_date.date()}")

train_dates = pd.date_range('2019-01-01', '2019-05-31')
test_dates = pd.date_range('2019-06-01', '2019-06-30')
train_dates = train_dates.intersection(all_dates_in_data)
test_dates = test_dates.intersection(all_dates_in_data)
print(f"训练集日期数: {len(train_dates)}, 测试集日期数: {len(test_dates)}")

# ---------- 8. 天气编码 ----------
weather_encoder = LabelEncoder()
weather_daily['weather_code_enc'] = weather_encoder.fit_transform(weather_daily['weather_code'].astype(str))

# 预加载POI静态摩擦向量（F1,F3,F4）
poi_friction_static = {}
for _, row in poi_friction.iterrows():
    pid = row['poi_id']
    poi_friction_static[pid] = np.array([row['F1'], row['F3'], row['F4']], dtype=np.float32)

In [ ]:
# ---------- 9. 辅助函数：构建单日超图（4类动态相似度超边）----------
def build_daily_hypergraph(date, scenic_static, hotel_nodes, poi_nodes,
                           scenic_friction, hotel_friction_daily,
                           weather_daily, search_daily,
                           scenic_id_to_hotel_indices, scenic_id_to_poi_indices,
                           poi_friction_static):
    data = HeteroData()
    num_scenic = len(scenic_static)
    num_hotel = len(hotel_nodes)
    num_poi = len(poi_nodes)

    # ----- 1. 景区节点特征（静态+动态）-----
    scenic_fric_today = scenic_friction[scenic_friction['date'] == date]
    # 获取每个景区的F1-F9
    scenic_fric_array = np.zeros((num_scenic, 9), dtype=np.float32)
    for _, row in scenic_static.iterrows():
        sid = row['scenic_id']
        idx = row['node_idx']
        vals = scenic_fric_today.loc[scenic_fric_today['scenic_id'] == sid, ['F1','F2','F3','F4','F5','F6','F7','F8','F9']].values
        if len(vals) > 0:
            scenic_fric_array[idx] = vals[0]
    f2_vals = scenic_fric_array[:, 1]   # F2

    search_today = search_daily[search_daily['date'] == date]
    search_vals = np.zeros(num_scenic)
    for _, row in scenic_static.iterrows():
        sid = row['scenic_id']
        idx = row['node_idx']
        val = search_today.loc[search_today['scenic_id'] == sid, 'search_lag'].values
        search_vals[idx] = val[0] if len(val) > 0 else 0.0

    weather_today = weather_daily[weather_daily['date'] == date]
    if len(weather_today) == 0:
        weather_feat = np.zeros(8)
    else:
        weather_cols = ['is_weekend', 'is_holiday', 'weather_code_enc',
                        'tem_high', 'tem_low', 'precipitation_sum',
                        'wind_speed_10m_mean', 'relative_humidity_2m_mean']
        weather_feat = weather_today[weather_cols].fillna(0).values[0].astype(np.float32)
    weather_feat_tiled = np.tile(weather_feat, (num_scenic, 1))
    scenic_feat = np.concatenate([scenic_static_feat.numpy(),
                                  f2_vals.reshape(-1,1),
                                  search_vals.reshape(-1,1),
                                  weather_feat_tiled], axis=1)
    data['scenic'].x = torch.tensor(scenic_feat, dtype=torch.float)

    # ----- 2. 酒店节点特征（静态）-----
    data['hotel'].x = hotel_static_feat.clone()

    # ----- 3. POI节点特征（静态）-----
    data['poi'].x = poi_static_feat.clone()

    # ----- 准备每日摩擦数据用于相似度计算 -----
    # 酒店摩擦 F5-F9
    hotel_fric_today = hotel_friction_daily[hotel_friction_daily['date'] == date]
    hotel_fric_array = np.zeros((num_hotel, 5), dtype=np.float32)
    for i, (_, row) in enumerate(hotel_nodes.iterrows()):
        hid = row['hotel_id']
        vals = hotel_fric_today.loc[hotel_fric_today['hotel_id'] == hid, ['F5','F6','F7','F8','F9']].values
        if len(vals) > 0:
            hotel_fric_array[i] = vals[0]
    # POI静态摩擦 (F1,F3,F4) – 预先存入数组
    poi_fric_array = np.zeros((num_poi, 3), dtype=np.float32)
    for i, (_, row) in enumerate(poi_nodes.iterrows()):
        pid = row['node_id']
        if pid in poi_friction_static:
            poi_fric_array[i] = poi_friction_static[pid]


        # ========== 1. 事件异质超边 (hyper_event): 景区 + 周边符合条件的酒店和POI ==========
    # 超边特征：所有连接酒店和POI摩擦的聚合（拼接：酒店5维 + POI3维 = 8维）
    hyper_event_nodes = []
    hyper_event_to_scenic = []      # 超边 -> 景区
    hyper_event_to_hotel = []       # 超边 -> 酒店
    hyper_event_to_poi = []         # 超边 -> POI

    for _, srow in scenic_static.iterrows():
        sid = srow['scenic_id']
        s_idx = srow['node_idx']
        s_fric = scenic_fric_array[s_idx]

        # 候选酒店（地理距离≤2km）
        hotel_candidates = scenic_id_to_hotel_indices.get(sid, [])
        # 候选POI（地理距离≤1.5km）
        poi_candidates = scenic_id_to_poi_indices.get(sid, [])

        # 景区F5-F9 vs 酒店F5-F9 (余弦相似度 > 0.7)
        s_f5f9 = s_fric[4:9]
        good_hotels = []
        for h_idx in hotel_candidates:
            h_fric = hotel_fric_array[h_idx]
            if np.isnan(h_fric).any():
                continue
            sim = np.dot(s_f5f9, h_fric) / (np.linalg.norm(s_f5f9) * np.linalg.norm(h_fric) + 1e-8)
            if sim > 0.7:
                good_hotels.append(h_idx)

        # 景区F1,F3,F4 vs POI F1,F3,F4 (余弦相似度 > 0.7)
        s_f134 = s_fric[[0,2,3]]
        good_pois = []
        for p_idx in poi_candidates:
            p_fric = poi_fric_array[p_idx]
            if np.isnan(p_fric).any():
                continue
            sim = np.dot(s_f134, p_fric) / (np.linalg.norm(s_f134) * np.linalg.norm(p_fric) + 1e-8)
            if sim > 0.7:
                good_pois.append(p_idx)

        if len(good_hotels) == 0 and len(good_pois) == 0:
            continue

        hyper_idx = len(hyper_event_nodes)
        # 超边特征：酒店5维 + POI3维 拼接
        hotel_mean = np.mean([hotel_fric_array[h] for h in good_hotels], axis=0) if good_hotels else np.zeros(5)
        poi_mean = np.mean([poi_fric_array[p] for p in good_pois], axis=0) if good_pois else np.zeros(3)
        feat = np.concatenate([hotel_mean, poi_mean])   # 8维
        hyper_event_nodes.append(feat)

        hyper_event_to_scenic.append([hyper_idx, s_idx])
        for h_idx in good_hotels:
            hyper_event_to_hotel.append([hyper_idx, h_idx])
        for p_idx in good_pois:
            hyper_event_to_poi.append([hyper_idx, p_idx])

    # 处理空超边情况
    if len(hyper_event_nodes) == 0:
        data['hyper_event'].x = torch.zeros((0, 8), dtype=torch.float)
        data['hyper_event', 'rev_to_scenic', 'scenic'].edge_index = torch.zeros((2,0), dtype=torch.long)
        data['scenic', 'to_hyper_event', 'hyper_event'].edge_index = torch.zeros((2,0), dtype=torch.long)
        data['hyper_event', 'rev_to_hotel', 'hotel'].edge_index = torch.zeros((2,0), dtype=torch.long)
        data['hotel', 'to_hyper_event', 'hyper_event'].edge_index = torch.zeros((2,0), dtype=torch.long)
        data['hyper_event', 'rev_to_poi', 'poi'].edge_index = torch.zeros((2,0), dtype=torch.long)
        data['poi', 'to_hyper_event', 'hyper_event'].edge_index = torch.zeros((2,0), dtype=torch.long)
    else:
        data['hyper_event'].x = torch.tensor(np.array(hyper_event_nodes), dtype=torch.float)
        if hyper_event_to_scenic:
            edge_scenic = torch.tensor(hyper_event_to_scenic, dtype=torch.long).t()
            data['hyper_event', 'rev_to_scenic', 'scenic'].edge_index = edge_scenic
            data['scenic', 'to_hyper_event', 'hyper_event'].edge_index = edge_scenic.flip(0)
        if hyper_event_to_hotel:
            edge_hotel = torch.tensor(hyper_event_to_hotel, dtype=torch.long).t()
            data['hyper_event', 'rev_to_hotel', 'hotel'].edge_index = edge_hotel
            data['hotel', 'to_hyper_event', 'hyper_event'].edge_index = edge_hotel.flip(0)
        if hyper_event_to_poi:
            edge_poi = torch.tensor(hyper_event_to_poi, dtype=torch.long).t()
            data['hyper_event', 'rev_to_poi', 'poi'].edge_index = edge_poi
            data['poi', 'to_hyper_event', 'hyper_event'].edge_index = edge_poi.flip(0)

    # ========== 2. 景区-景区超边 (hyper_scenic_scenic) ==========
    hyper_ss_nodes = []
    hyper_ss_edges = []   # [hyper_idx, scenic_idx] 每对两个记录
    for i in range(num_scenic):
        for j in range(i+1, num_scenic):
            s_fric_i = scenic_fric_array[i]
            s_fric_j = scenic_fric_array[j]
            if np.isnan(s_fric_i).any() or np.isnan(s_fric_j).any():
                continue
            sim = np.dot(s_fric_i, s_fric_j) / (np.linalg.norm(s_fric_i) * np.linalg.norm(s_fric_j) + 1e-8)
            if sim > 0.7:
                hyper_idx = len(hyper_ss_nodes)
                hyper_ss_nodes.append((s_fric_i + s_fric_j) / 2.0)   # 9维
                hyper_ss_edges.append([hyper_idx, i])
                hyper_ss_edges.append([hyper_idx, j])

    if len(hyper_ss_nodes) == 0:
        data['hyper_scenic_scenic'].x = torch.zeros((0, 9), dtype=torch.float)
        data['hyper_scenic_scenic', 'rev_to_scenic', 'scenic'].edge_index = torch.zeros((2,0), dtype=torch.long)
        data['scenic', 'to_hyper_ss', 'hyper_scenic_scenic'].edge_index = torch.zeros((2,0), dtype=torch.long)
    else:
        data['hyper_scenic_scenic'].x = torch.tensor(np.array(hyper_ss_nodes), dtype=torch.float)
        edges = torch.tensor(hyper_ss_edges, dtype=torch.long).t()
        data['hyper_scenic_scenic', 'rev_to_scenic', 'scenic'].edge_index = edges
        data['scenic', 'to_hyper_ss', 'hyper_scenic_scenic'].edge_index = edges.flip(0)

    # ========== 3. 酒店-酒店超边 (hyper_hotel_hotel) ==========
    hyper_hh_nodes = []
    hyper_hh_edges = []
    for i in range(num_hotel):
        for j in range(i+1, num_hotel):
            h_fric_i = hotel_fric_array[i]
            h_fric_j = hotel_fric_array[j]
            if np.isnan(h_fric_i).any() or np.isnan(h_fric_j).any():
                continue
            sim = np.dot(h_fric_i, h_fric_j) / (np.linalg.norm(h_fric_i) * np.linalg.norm(h_fric_j) + 1e-8)
            if sim > 0.7:
                hyper_idx = len(hyper_hh_nodes)
                hyper_hh_nodes.append((h_fric_i + h_fric_j) / 2.0)   # 5维
                hyper_hh_edges.append([hyper_idx, i])
                hyper_hh_edges.append([hyper_idx, j])

    if len(hyper_hh_nodes) == 0:
        data['hyper_hotel_hotel'].x = torch.zeros((0, 5), dtype=torch.float)
        data['hyper_hotel_hotel', 'rev_to_hotel', 'hotel'].edge_index = torch.zeros((2,0), dtype=torch.long)
        data['hotel', 'to_hyper_hh', 'hyper_hotel_hotel'].edge_index = torch.zeros((2,0), dtype=torch.long)
    else:
        data['hyper_hotel_hotel'].x = torch.tensor(np.array(hyper_hh_nodes), dtype=torch.float)
        edges = torch.tensor(hyper_hh_edges, dtype=torch.long).t()
        data['hyper_hotel_hotel', 'rev_to_hotel', 'hotel'].edge_index = edges
        data['hotel', 'to_hyper_hh', 'hyper_hotel_hotel'].edge_index = edges.flip(0)

    return data

# ---------- 10. 构建每日图（所有日期）----------
all_dates = train_dates.union(test_dates).sort_values()
daily_graphs = []
for date in tqdm(all_dates, desc="构建每日超图"):
    g = build_daily_hypergraph(date, scenic_static, hotel_nodes, poi_nodes,
                               scenic_friction, hotel_friction_daily,
                               weather_daily, search_daily,
                               scenic_id_to_hotel_indices, scenic_id_to_poi_indices,
                               poi_friction_static)
    daily_graphs.append(g)
print(f"每日图总数: {len(daily_graphs)}")


In [ ]:
# ---------- 11. 生成序列样本（滞后特征）----------
def create_sequences_with_lag(graphs, target_df, scenic_ids, seq_len, pred_horizon, date_list):
    X, y = [], []
    graph_date_index = {date: i for i, date in enumerate(all_dates)}
    visit_pivot = target_df.pivot(index='date', columns='scenic_id', values='visit_index')
    visit_pivot = visit_pivot.reindex(date_list)
    prev_date_map = {}
    for i, d in enumerate(date_list):
        if i > 0:
            prev_date_map[d] = date_list[i-1]
        else:
            prev_date_map[d] = None

    for i in range(len(date_list) - seq_len - pred_horizon + 1):
        seq_dates = date_list[i:i+seq_len]
        target_date = date_list[i+seq_len+pred_horizon-1]
        if target_date not in visit_pivot.index:
            continue
        y_vals = visit_pivot.loc[target_date, scenic_ids].values
        if np.any(pd.isna(y_vals)):
            continue

        x_seq = []
        for t, d in enumerate(seq_dates):
            g = graphs[graph_date_index[d]]
            scenic_x_orig = g['scenic'].x.numpy()
            prev_d = prev_date_map.get(d)
            if prev_d is not None and prev_d in visit_pivot.index:
                lag_vals = visit_pivot.loc[prev_d, scenic_ids].values
            else:
                lag_vals = np.zeros(len(scenic_ids))
            lag_tensor = torch.tensor(lag_vals, dtype=torch.float).unsqueeze(1)
            new_scenic_x = torch.cat([torch.tensor(scenic_x_orig, dtype=torch.float), lag_tensor], dim=1)
            new_g = HeteroData()
            for node_type in g.node_types:
                if node_type == 'scenic':
                    new_g[node_type].x = new_scenic_x
                else:
                    new_g[node_type].x = g[node_type].x.clone()
            for edge_type, edge_idx in g.edge_index_dict.items():
                new_g[edge_type].edge_index = edge_idx.clone()
            x_seq.append(new_g)
        X.append(x_seq)
        y.append(torch.tensor(y_vals, dtype=torch.float))
    return X, y

X_train_raw, y_train_raw = create_sequences_with_lag(daily_graphs, scenic_visit_daily, valid_scenic_ids,
                                                     SEQ_LEN, PRED_HORIZON, train_dates.tolist())
X_test_raw, y_test_raw = create_sequences_with_lag(daily_graphs, scenic_visit_daily, valid_scenic_ids,
                                                   SEQ_LEN, PRED_HORIZON, test_dates.tolist())
print(f"原始训练样本数: {len(X_train_raw)}, 测试样本数: {len(X_test_raw)}")

# ---------- 12. 标准化（在划分之后）----------
y_train = y_train_raw
y_test = y_test_raw

all_scenic_dynamic = []
for seq in X_train_raw:
    for g in seq:
        scenic_x = g['scenic'].x.numpy()
        dynamic_part = scenic_x[:, 1:]  # 去掉第一列静态grade_code
        all_scenic_dynamic.append(dynamic_part)
all_scenic_dynamic = np.vstack(all_scenic_dynamic)
dynamic_scaler = StandardScaler()
dynamic_scaler.fit(all_scenic_dynamic)

def scale_scenic_features(graph_seq, dynamic_scaler):
    new_seq = []
    for g in graph_seq:
        new_g = HeteroData()
        for node_type in g.node_types:
            if node_type == 'scenic':
                x = g[node_type].x.numpy()
                static_part = x[:, :1]
                dynamic_part = x[:, 1:]
                dynamic_norm = dynamic_scaler.transform(dynamic_part)
                x_new = np.concatenate([static_part, dynamic_norm], axis=1)
                new_g[node_type].x = torch.tensor(x_new, dtype=torch.float)
            else:
                new_g[node_type].x = g[node_type].x.clone()
        for edge_type, edge_idx in g.edge_index_dict.items():
            new_g[edge_type].edge_index = edge_idx.clone()
        new_seq.append(new_g)
    return new_seq

X_train = [scale_scenic_features(seq, dynamic_scaler) for seq in X_train_raw]
X_test = [scale_scenic_features(seq, dynamic_scaler) for seq in X_test_raw]
print(f"标准化后训练样本数: {len(X_train)}, 测试样本数: {len(X_test)}")
print(f"景区节点特征最终维度: {X_train[0][0]['scenic'].x.shape[1]}")
input_dim_scenic = X_train[0][0]['scenic'].x.shape[1]
num_scenics = len(valid_scenic_ids)

def move_to_device(graph_seq_list, device):
    return [[g.to(device) for g in seq] for seq in graph_seq_list]

X_train = move_to_device(X_train, DEVICE)
X_test  = move_to_device(X_test, DEVICE)
y_train = [y.to(DEVICE) for y in y_train]
y_test  = [y.to(DEVICE) for y in y_test]

print("Sample data device:", X_train[0][0]['scenic'].x.device)

In [ ]:
# ---------- 13. 超图模型定义（支持五类超边）----------  hyper_pp_dim

class HypergraphSTModel(nn.Module):
    def __init__(self, in_dim_scenic, in_dim_hotel, in_dim_poi,
             hyper_event_dim, hyper_ss_dim, hyper_hh_dim, hidden_dim, num_scenics):
        super().__init__()
        # 节点特征线性变换
        self.scenic_lin = nn.Linear(in_dim_scenic, hidden_dim)
        self.hotel_lin = nn.Linear(in_dim_hotel, hidden_dim)
        self.poi_lin = nn.Linear(in_dim_poi, hidden_dim)
        # 超边节点特征线性变换
        self.hyper_event_lin = nn.Linear(hyper_event_dim, hidden_dim)
        self.hyper_ss_lin = nn.Linear(hyper_ss_dim, hidden_dim)
        self.hyper_hh_lin = nn.Linear(hyper_hh_dim, hidden_dim)

        # 节点->超边 线性层（用于聚合时对节点特征变换）
        self.node_to_hyper = nn.ModuleDict({
            'hyper_event': nn.Linear(hidden_dim, hidden_dim),
            'hyper_scenic_scenic': nn.Linear(hidden_dim, hidden_dim),
            'hyper_hotel_hotel': nn.Linear(hidden_dim, hidden_dim),
        })
        # 超边->节点 线性层（用于更新节点时对超边特征变换）
        self.hyper_to_node = nn.ModuleDict({
            'scenic': nn.Linear(hidden_dim, hidden_dim),
            'hotel': nn.Linear(hidden_dim, hidden_dim),
            'poi': nn.Linear(hidden_dim, hidden_dim),
        })

        self.gru = nn.GRU(hidden_dim, 32, batch_first=True)
        self.fc = nn.Linear(32, num_scenics)
        self.dropout_spatial = nn.Dropout(p=0.5)
        self.dropout_temporal = nn.Dropout(p=0.5)

        self.gru = nn.GRU(hidden_dim, 32, batch_first=True)
        self.fc = nn.Linear(32, num_scenics)
        self.dropout_spatial = nn.Dropout(p=0.5)
        self.dropout_temporal = nn.Dropout(p=0.5)
    
    def forward(self, graph_seq):
        seq_len = len(graph_seq)
        scenic_hidden_seq = []

        for t, data in enumerate(graph_seq):
            x_scenic = F.relu(self.scenic_lin(data['scenic'].x))
            x_hotel   = F.relu(self.hotel_lin(data['hotel'].x))
            x_poi     = F.relu(self.poi_lin(data['poi'].x))
            x_hyper_event = F.relu(self.hyper_event_lin(data['hyper_event'].x))
            x_hyper_ss    = F.relu(self.hyper_ss_lin(data['hyper_scenic_scenic'].x))
            x_hyper_hh    = F.relu(self.hyper_hh_lin(data['hyper_hotel_hotel'].x))
            
            node_dict = {
                'scenic': x_scenic,
                'hotel': x_hotel,
                'poi': x_poi,
                'hyper_event': x_hyper_event,
                'hyper_scenic_scenic': x_hyper_ss,
                'hyper_hotel_hotel': x_hyper_hh,
            }

            # ---------- 节点 -> 超边 ----------
            # 1. hyper_event: 汇聚 scenic, hotel, poi
            hyper_type = 'hyper_event'
            all_nodes = []
            all_edges = []
            for src_type in ['scenic', 'hotel', 'poi']:
                key = (src_type, f'to_hyper_event', hyper_type)
                if key in data.edge_index_dict:
                    edge = data[key].edge_index
                    if edge.size(1) > 0:
                        all_edges.append(edge)
                        all_nodes.append(node_dict[src_type])
            if all_edges:
                total_edges = torch.cat(all_edges, dim=1)
                total_nodes = torch.cat(all_nodes, dim=0)
                N = total_nodes.size(0)
                E = node_dict[hyper_type].size(0)
                indices = total_edges[[1, 0]]
                values = torch.ones(total_edges.size(1), device=total_nodes.device)
                H = torch.sparse_coo_tensor(indices, values, (N, E), device=total_nodes.device).coalesce()
                hyper_agg = torch.sparse.mm(H.t(), total_nodes)
                deg = torch.sparse.sum(H, dim=0).to_dense().unsqueeze(1)
                hyper_agg = hyper_agg / (deg + 1e-8)
                node_dict[hyper_type] = hyper_agg + node_dict[hyper_type]

            # 2. hyper_scenic_scenic: 只汇聚 scenic
            hyper_type = 'hyper_scenic_scenic'
            key = ('scenic', 'to_hyper_ss', hyper_type)
            if key in data.edge_index_dict:
                edge = data[key].edge_index
                if edge.size(1) > 0:
                    indices = edge[[1, 0]]
                    N = node_dict['scenic'].size(0)
                    E = node_dict[hyper_type].size(0)
                    values = torch.ones(edge.size(1), device=node_dict['scenic'].device)
                    H = torch.sparse_coo_tensor(indices, values, (N, E), device=node_dict['scenic'].device).coalesce()
                    hyper_agg = torch.sparse.mm(H.t(), node_dict['scenic'])
                    deg = torch.sparse.sum(H, dim=0).to_dense().unsqueeze(1)
                    hyper_agg = hyper_agg / (deg + 1e-8)
                    node_dict[hyper_type] = hyper_agg + node_dict[hyper_type]

            # 3. hyper_hotel_hotel: 只汇聚 hotel
            hyper_type = 'hyper_hotel_hotel'
            key = ('hotel', 'to_hyper_hh', hyper_type)
            if key in data.edge_index_dict:
                edge = data[key].edge_index
                if edge.size(1) > 0:
                    indices = edge[[1, 0]]
                    N = node_dict['hotel'].size(0)
                    E = node_dict[hyper_type].size(0)
                    values = torch.ones(edge.size(1), device=node_dict['hotel'].device)
                    H = torch.sparse_coo_tensor(indices, values, (N, E), device=node_dict['hotel'].device).coalesce()
                    hyper_agg = torch.sparse.mm(H.t(), node_dict['hotel'])
                    deg = torch.sparse.sum(H, dim=0).to_dense().unsqueeze(1)
                    hyper_agg = hyper_agg / (deg + 1e-8)
                    node_dict[hyper_type] = hyper_agg + node_dict[hyper_type]

            # ---------- 超边 -> 节点 ----------
            # 景区节点从 hyper_event 和 hyper_scenic_scenic 接收信息
            scenic_updates = []
            for hyper_type in ['hyper_event', 'hyper_scenic_scenic']:
                rev_key = (hyper_type, 'rev_to_scenic', 'scenic')
                if rev_key in data.edge_index_dict:
                    edge = data[rev_key].edge_index
                    if edge.size(1) > 0:
                        indices = edge[[1, 0]]
                        N = node_dict['scenic'].size(0)
                        E = node_dict[hyper_type].size(0)
                        values = torch.ones(edge.size(1), device=node_dict['scenic'].device)
                        Ht = torch.sparse_coo_tensor(indices, values, (N, E), device=node_dict['scenic'].device).coalesce()
                        node_agg = torch.sparse.mm(Ht, node_dict[hyper_type])
                        deg = torch.sparse.sum(Ht, dim=1).to_dense().unsqueeze(1)
                        node_agg = node_agg / (deg + 1e-8)
                        scenic_updates.append(node_agg)
            if scenic_updates:
                node_dict['scenic'] = torch.stack(scenic_updates, dim=0).sum(dim=0) + node_dict['scenic']

            # 酒店节点从 hyper_event 和 hyper_hotel_hotel 接收信息
            hotel_updates = []
            for hyper_type in ['hyper_event', 'hyper_hotel_hotel']:
                rev_key = (hyper_type, 'rev_to_hotel', 'hotel')
                if rev_key in data.edge_index_dict:
                    edge = data[rev_key].edge_index
                    if edge.size(1) > 0:
                        indices = edge[[1, 0]]
                        N = node_dict['hotel'].size(0)
                        E = node_dict[hyper_type].size(0)
                        values = torch.ones(edge.size(1), device=node_dict['hotel'].device)
                        Ht = torch.sparse_coo_tensor(indices, values, (N, E), device=node_dict['hotel'].device).coalesce()
                        node_agg = torch.sparse.mm(Ht, node_dict[hyper_type])
                        deg = torch.sparse.sum(Ht, dim=1).to_dense().unsqueeze(1)
                        node_agg = node_agg / (deg + 1e-8)
                        hotel_updates.append(node_agg)
            if hotel_updates:
                node_dict['hotel'] = torch.stack(hotel_updates, dim=0).sum(dim=0) + node_dict['hotel']


            scenic_emb = node_dict['scenic']
            scenic_emb = self.dropout_spatial(scenic_emb)
            scenic_hidden_seq.append(scenic_emb)

        stacked = torch.stack(scenic_hidden_seq, dim=0)
        pooled = stacked.mean(dim=1).unsqueeze(0)
        _, h_n = self.gru(pooled)
        h_n_out = self.dropout_temporal(h_n.squeeze(0))
        out = self.fc(h_n_out)
        return out.squeeze(0)

# 提取各特征维度（必须在 X_train 标准化之后）
in_dim_scenic = X_train[0][0]['scenic'].x.shape[1]
in_dim_hotel = X_train[0][0]['hotel'].x.shape[1]
in_dim_poi = X_train[0][0]['poi'].x.shape[1]
hyper_event_dim = X_train[0][0]['hyper_event'].x.shape[1]          # 应为 8
hyper_ss_dim = X_train[0][0]['hyper_scenic_scenic'].x.shape[1]    # 9
hyper_hh_dim = X_train[0][0]['hyper_hotel_hotel'].x.shape[1]      # 5

model = HypergraphSTModel(in_dim_scenic, in_dim_hotel, in_dim_poi,
                          hyper_event_dim, hyper_ss_dim, hyper_hh_dim,
                          HIDDEN_CHANNELS, num_scenics).to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)
loss_fn = nn.MSELoss()

# ---------- 14. 评估函数 ----------
def compute_metrics(y_true, y_pred, num_scenics):
    # 转换为 numpy 并过滤 NaN
    if torch.is_tensor(y_true):
        y_true = y_true.cpu().numpy()
    if torch.is_tensor(y_pred):
        y_pred = y_pred.cpu().numpy()
    
    # 展平或保持原形状（假设 y_true, y_pred 形状为 (n_samples, n_scenics)）
    y_true_flat = y_true.flatten()
    y_pred_flat = y_pred.flatten()
    
    # 移除 NaN 对
    mask = ~(np.isnan(y_true_flat) | np.isnan(y_pred_flat))
    y_true_clean = y_true_flat[mask]
    y_pred_clean = y_pred_flat[mask]
    
    if len(y_true_clean) == 0:
        return {'MSE': np.nan, 'RMSE': np.nan, 'MAE': np.nan, 'MAPE': np.nan, 'R2': np.nan}
    
    mse = mean_squared_error(y_true_clean, y_pred_clean)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true_clean, y_pred_clean)
    
    # MAPE
    mask_nonzero = y_true_clean != 0
    if mask_nonzero.sum() == 0:
        mape = np.nan
    else:
        mape = np.mean(np.abs((y_true_clean[mask_nonzero] - y_pred_clean[mask_nonzero]) / y_true_clean[mask_nonzero])) * 100
    
    r2 = r2_score(y_true_clean, y_pred_clean)
    
    return {'MSE': mse, 'RMSE': rmse, 'MAE': mae, 'MAPE': mape, 'R2': r2}

# ---------- 15. 训练循环 ----------
train_losses = []
test_metrics_records = []
best_mae = np.inf
wait = 0

for epoch in range(1, EPOCHS+1):
    model.train()
    total_loss = 0
    num_batches = 0
    indices = np.random.permutation(len(X_train))
    pbar = tqdm(range(0, len(X_train), BATCH_SIZE), desc=f"Epoch {epoch}/{EPOCHS}", unit="batch")
    for i in pbar:
        batch_idx = indices[i:i+BATCH_SIZE]
        batch_loss = 0.0
        for idx in batch_idx:
            x_seq = X_train[idx]
            y_true = y_train[idx].to(DEVICE)
            pred = model(x_seq)
            loss = loss_fn(pred, y_true)
            batch_loss += loss
        batch_loss /= len(batch_idx)
        optimizer.zero_grad()
        batch_loss.backward()
        # torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5) #梯度剪裁
        optimizer.step()
        total_loss += batch_loss.item()
        num_batches += 1
        pbar.set_postfix({'loss': batch_loss.item()})
    avg_train_loss = total_loss / num_batches
    train_losses.append(avg_train_loss)

    # 测试评估
    model.eval()
    all_preds_norm = []
    all_trues_norm = []
    with torch.no_grad():
        for x_seq, y_true in zip(X_test, y_test):
            pred = model(x_seq).cpu()
            all_preds_norm.append(pred)
            all_trues_norm.append(y_true.cpu())
    all_preds_norm = torch.stack(all_preds_norm)
    all_trues_norm = torch.stack(all_trues_norm)
    mask = ~(torch.isnan(all_preds_norm).any(dim=1) | torch.isnan(all_trues_norm).any(dim=1))
    all_preds_norm = all_preds_norm[mask]
    all_trues_norm = all_trues_norm[mask]
    if len(all_trues_norm) == 0:
        metrics = {'MSE': np.nan, 'RMSE': np.nan, 'MAE': np.nan, 'MAPE': np.nan, 'R2': np.nan}
    else:
        metrics = compute_metrics(all_trues_norm, all_preds_norm, num_scenics)
    test_metrics_records.append(metrics)
    print(f"Epoch {epoch:2d} | Train Loss: {avg_train_loss:.6f} | "
          f"Test MSE: {metrics['MSE']:.4f} | RMSE: {metrics['RMSE']:.4f} | "
          f"MAE: {metrics['MAE']:.4f} | MAPE: {metrics['MAPE']:.2f}% | R2: {metrics['R2']:.4f}")

    if metrics['MAE'] < best_mae:
        best_mae = metrics['MAE']
        torch.save(model.state_dict(), MODEL_SAVE_PATH)
        print(f"  -> 保存最佳模型 (MAE={best_mae:.4f})")
        wait = 0
    else:
        wait += 1
        if wait >= PATIENCE:
            print(f"早停于 epoch {epoch}")
            break

# ---------- 16. 保存记录与绘图 ----------
loss_df = pd.DataFrame({'epoch': range(1, len(train_losses)+1), 'train_loss': train_losses})
loss_df.to_csv(LOSS_LOG_PATH, index=False)
metrics_df = pd.DataFrame(test_metrics_records)
metrics_df.insert(0, 'epoch', range(1, len(test_metrics_records)+1))
metrics_df.to_csv(METRICS_LOG_PATH, index=False)

plt.figure(figsize=(10,5))
plt.plot(range(1, len(train_losses)+1), train_losses, marker='o', label='Train Loss')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.title('Training Loss Curve')
plt.legend(); plt.grid(True)
plt.savefig(LOSS_CURVE_PATH, dpi=150); plt.close()

plt.figure(figsize=(12,6))
plt.subplot(2,2,1); plt.plot(metrics_df['epoch'], metrics_df['MSE'], color='r', label='MSE')
plt.xlabel('Epoch'); plt.ylabel('MSE'); plt.legend(); plt.grid(True)
plt.subplot(2,2,2); plt.plot(metrics_df['epoch'], metrics_df['RMSE'], color='g', label='RMSE')
plt.xlabel('Epoch'); plt.ylabel('RMSE'); plt.legend(); plt.grid(True)
plt.subplot(2,2,3); plt.plot(metrics_df['epoch'], metrics_df['MAE'], color='b', label='MAE')
plt.xlabel('Epoch'); plt.ylabel('MAE'); plt.legend(); plt.grid(True)
plt.subplot(2,2,4); plt.plot(metrics_df['epoch'], metrics_df['R2'], color='m', label='R²')
plt.xlabel('Epoch'); plt.ylabel('R²'); plt.legend(); plt.grid(True)
plt.tight_layout()
plt.savefig(METRICS_CURVE_PATH, dpi=150); plt.close()

print(f"\n训练完成！最佳模型保存至 {MODEL_SAVE_PATH}")
print(f"损失记录: {LOSS_LOG_PATH}\n指标记录: {METRICS_LOG_PATH}")
print(f"损失曲线: {LOSS_CURVE_PATH}\n指标曲线: {METRICS_CURVE_PATH}")

# 加载最佳模型输出最终指标
model.load_state_dict(torch.load(MODEL_SAVE_PATH, map_location=DEVICE))
model.eval()
all_preds, all_trues = [], []
with torch.no_grad():
    for x_seq, y_true in zip(X_test, y_test):
        x_seq = [g.to(DEVICE) for g in x_seq]
        pred = model(x_seq).cpu()
        all_preds.append(pred.numpy())
        all_trues.append(y_true.cpu().numpy())
all_preds = np.concatenate(all_preds)
all_trues = np.concatenate(all_trues)
best_metrics = compute_metrics(all_trues, all_preds, num_scenics)
print("\n最佳模型在测试集上的指标:")
print(f"MSE: {best_metrics['MSE']:.4f}, RMSE: {best_metrics['RMSE']:.4f}, MAE: {best_metrics['MAE']:.4f}, MAPE: {best_metrics['MAPE']:.2f}%, R2: {best_metrics['R2']:.4f}")

In [ ]:
# ---------- 1. 随机种子（可复现）----------
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# ---------- 2. 全局参数 ----------
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEQ_LEN = 3
PRED_HORIZON = 1
BATCH_SIZE = 16
EPOCHS = 50
HIDDEN_CHANNELS = 4          # 超图卷积后的隐层维度
NUM_HYPEREDGE_FEAT = 9       # 超边特征维度（F1-F9）
PATIENCE = 10
LEARNING_RATE = 0.05

LOSS_LOG_PATH = './Hyper2/loss_log_frihyper10.csv'
METRICS_LOG_PATH = './Hyper2/metrics_log_frihyper10.csv'
LOSS_CURVE_PATH = './Hyper2/loss_curve_frihyper10.png'
METRICS_CURVE_PATH = './Hyper2/metrics_curve_frihyper10.png'
MODEL_SAVE_PATH = './Hyper2/best_model_frihyper10.pth'